# Fitting the Crab Spectrum with the Binned Relative-Coordinates Response and the Histogram Background

## Introduction

This notebook demonstrates an unbinned spectral analysis of the Crab, using the histogram-based response (`IRFRelativeHistUnpolarized`) and the histogram-based background (`HistBackgroundTemplate` and `FreeNormHistBackgroundDensity`), neither of which needs a neural network. The classes are described in
[example_grb_fit_relative_hist_response.ipynb](../grb/example_grb_fit_relative_hist_response.ipynb), and the rest of the unbinned fitting machinery in
[example_crab_fit_normalizing_flows.ipynb](example_crab_fit_normalizing_flows.ipynb) and [example_grb_fit_normalizing_flows.ipynb](../grb/example_grb_fit_normalizing_flows.ipynb). Here we only cover what is different.

The data is the DC4 mock dataset (week 4), which we treat as if it were real data: it contains the Crab, all the other sources and the background. We analyze 3 hours of it.

The background template is built from the data itself (from the 3 months of the mock dataset, or from this week). Unlike the GRB, the Crab is a continuous source, so we cannot remove an on-time window. We don't need to: as the spacecraft rotates, the Crab moves across the Compton data space in the spacecraft frame, so its photons are washed out in the $\phi$ and $\Psi\chi$ histograms. The Crab is only ~1% of the events in this window, so the background model matters much more than for the GRB.

## Example

### Choose the response mode and the background template source

In [ ]:
irf_mode = "hist_simple"  # one of "hist_simple", "hist_nn", "nn"

bkg_template_source = "wasabi"  # one of "wasabi", "week" (see "Background template" below)


### Basic Setup

In [ ]:
from pathlib import Path
from cosipy.util import fetch_wasabi_file
from cosipy.spacecraftfile import SpacecraftHistory
from astropy.time import Time
import urllib.request

import astropy.units as u
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np

from threeML import PointSource, Model, JointLikelihood, DataList
from astromodels import Parameter, Powerlaw

from cosipy.threeml.unbinned_model_folding import CachedUnbinnedThreeMLModelFolding
from cosipy.statistics import UnbinnedLikelihood
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.interfaces.expectation_interface import SumExpectationDensity

from cosipy.event_selection import DistanceSelector, ChainEventSelectors
from cosipy.event_selection.time_selection import TimeSelector
from cosipy.data_io.EmCDSUnbinnedData import (TimeTagEmCDSDistanceEventDataInSCFrameFromDC3Fits,
                                              TimeTagEmCDSDistanceEventDataInSCFrameFromArrays)

from cosipy.response.ml.NFResponse import NFResponse
from cosipy.response.ml.nf_instrument_response_function import UnpolarizedNFFarFieldInstrumentResponseFunction
from cosipy.response.relative_irf_hist import IRFRelativeHistUnpolarized

from cosipy.background_estimation import HistBackgroundTemplate, FreeNormHistBackgroundDensity
from histpy import Axes, Axis, HealpixAxis
from scoords import SpacecraftFrame
import healpy as hp

from cosipy.threeml.ml.optimized_unbinned_folding import UnbinnedThreeMLPointSourceResponseIRFAdaptive


The event data and spacecraft orientation are needed regardless of `irf_mode`; the
response file is not, so it's only fetched below for the mode actually selected above.

In [ ]:
data_path = Path("./") # Current path by default

week_data_path = data_path / "dc4_mock_dataset_week_4_unbinned_data_filtered_with_SAAcut.fits.gz"
fetch_wasabi_file('COSI-SMEX/DC4/Data/Mock_Dataset/dc4_mock_dataset_week_4_unbinned_data_filtered_with_SAAcut.fits.gz',
                  checksum = '034fa0046ea6116a85ee1ad1365cce72', output=str(week_data_path))

sc_orientation_path = data_path / "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
fetch_wasabi_file('COSI-SMEX/DC4/Data/Orientation/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits',
                  checksum = 'ca94ff1d7a73c1f41479aaf598807673', output=str(sc_orientation_path))


In [ ]:
if irf_mode == "nn":

    rsp_path = data_path / "unpolarized_nfresponse_v1-01.pt"
    fetch_wasabi_file('COSI-SMEX/DC4/Data/Responses/unpolarized_nfresponse_v1-01.pt',
                      checksum = 'bf2d0c16eac5954fb56489480c2602ca', output=str(rsp_path))

elif irf_mode == "hist_simple":

    hist_zip_path = data_path / "ResponseContinuum.area.relative.nonsparse_smoothing1p0.h5.zip"
    hist_path = data_path / "ResponseContinuum.area.relative.nonsparse_smoothing1p0.h5"
    fetch_wasabi_file('COSI-SMEX/develop/Data/Responses/ResponseContinuum.area.relative.nonsparse_smoothing1p0.h5.zip',
                      output=str(hist_zip_path), unzip=True,
                      checksum = 'bd2dfa700d0d382b052ea428a7eeabe1')

elif irf_mode == "hist_nn":

    hist_nn_zip_path = data_path / "relative_hist_irf_from_nf_response.h5.zip"
    hist_nn_path = data_path / "relative_hist_irf_from_nf_response.h5"
    fetch_wasabi_file('COSI-SMEX/develop/Data/Responses/relative_hist_irf_from_nf_response.h5.zip',
                      output=str(hist_nn_zip_path), unzip=True,
                      checksum = 'd9093daeaf56a386095a42ae40c6635e')

else:

    raise RuntimeError(f"irf_mode {irf_mode} is not supported.")


Define the observation duration here. For this tutorial we choose three hours.

In [ ]:
tstart = Time("2028-03-23 10:36:00.000")
tstop = Time("2028-03-23 13:36:00.000")
sc_orientation_full = SpacecraftHistory.open(sc_orientation_path)
sc_orientation = sc_orientation_full.select_interval(tstart, tstop)


`"hist_simple"` also needs a distance cut (first-two-hits distance > 1 cm): the MEGAlib
simulation it was built from used that same cut to avoid strip-discretization artifacts near the
detector edges, so the event data needs to match. `"hist_nn"` and `"nn"` don't need it, since
they aren't built using that cut.

I set >1cm because those events:
* Make the response very complicated to model, due to position discretization.
* Are very low quality anyway. Their corresponding ARM is very bad, so they don’t give you much resolution nor sensitivity.

Eventually we'll add the distance as a parameter of the IRF, but for now filtering out these is a net positive.

In [ ]:
time_selector = TimeSelector(tstart = sc_orientation.tstart, tstop = sc_orientation.tstop)

if irf_mode == "hist_simple":
    selector = ChainEventSelectors(DistanceSelector(min_distance=1 * u.cm), time_selector)
else:
    selector = time_selector

if bkg_template_source == "week":
    # We need the whole week to build the background template, so read it all and select from it
    week_data = TimeTagEmCDSDistanceEventDataInSCFrameFromDC3Fits([week_data_path])

    data = TimeTagEmCDSDistanceEventDataInSCFrameFromArrays(
        week_data.jd1, week_data.jd2, week_data.energy_keV,
        week_data.scattered_lon_rad_sc, week_data.scattered_lat_rad_sc,
        week_data.scattering_angle_rad, week_data.distance_cm,
        selection=selector)
else:
    data = TimeTagEmCDSDistanceEventDataInSCFrameFromDC3Fits([week_data_path], selection=selector)


In [ ]:
print(f"This analysis uses {data.nevents} Events")

### Background template

`HistBackgroundTemplate` holds the raw counts of four histograms, and the livetime they were accumulated over:
the rate as a function of time and energy, the Compton angle $\phi$ as a function of rocking angle and energy, and the
scattered direction $\Psi\chi$ as a function of rocking angle, energy and $\phi$. They are smoothed and
interpolated when evaluating the density.

- `"wasabi"`: load the template built from the 3 months of the DC4 mock dataset with
  `cosipy/background_estimation/scripts/HistBackgroundTemplate/build_hist_background_template.py`. It has
  14 times more statistics than the one from a single week. For `"hist_simple"`, we need the version
  built with the same distance cut as the data.
- `"week"`: build the template here from the same week of data that we are analyzing. This is quicker (about a minute),
  but noisier.

In [ ]:
if bkg_template_source == "wasabi":

    suffix = "_dist1cm" if irf_mode == "hist_simple" else ""
    bkg_template_zip_path = data_path / f"dc4_mock_bkg_template{suffix}.h5.zip"
    bkg_template_path = data_path / f"dc4_mock_bkg_template{suffix}.h5"
    checksum = "8bb361a5b9b3f145fe2b965006b2bd3c" if irf_mode == "hist_simple" else "fa1011ca70f8d583ac9b4fc08db73e66"

    fetch_wasabi_file(f'COSI-SMEX/develop/Data/Products_and_Templates/Background_Models/HistBackgroundTemplate/dc4_mock_bkg_template{suffix}.h5.zip',
                      output=str(bkg_template_zip_path), unzip=True,
                      checksum=checksum)

    template = HistBackgroundTemplate.open(bkg_template_path)

elif bkg_template_source == "week":

    # Same binning as build_hist_background_template.py
    time_bin = 300.
    time_start = np.floor(sc_orientation_full.tstart.utc.unix / time_bin) * time_bin
    ntime_bins = int(np.ceil((sc_orientation_full.tstop.utc.unix - time_start) / time_bin))

    rocking_axis = Axis([-90., 0., 90.], label='Rocking', unit=u.deg)

    rate_axes = Axes([Axis(time_start + time_bin * np.arange(ntime_bins + 1), label='Time'),
                      Axis(np.geomspace(100., 1e4, 30 + 1), scale='log', label='Em')])

    phi_axes = Axes([rocking_axis,
                     Axis(np.geomspace(100., 1e4, 30 + 1), scale='log', label='Em'),
                     Axis(np.deg2rad(np.arange(0., 180.1, 2.)), label='Phi')])

    psichi_axes = Axes([rocking_axis,
                        Axis(np.geomspace(100., 1e4, 8 + 1), scale='log', label='Em'),
                        Axis(np.deg2rad(np.arange(0., 180.1, 10.)), label='Phi'),
                        HealpixAxis(nside=32, scheme='ring', coordsys=SpacecraftFrame(), label='PsiChi')])

    template = HistBackgroundTemplate.empty(rate_axes, phi_axes, psichi_axes)

    if irf_mode == "hist_simple":
        template_events = TimeTagEmCDSDistanceEventDataInSCFrameFromArrays(
            week_data.jd1, week_data.jd2, week_data.energy_keV,
            week_data.scattered_lon_rad_sc, week_data.scattered_lat_rad_sc,
            week_data.scattering_angle_rad, week_data.distance_cm,
            selection=DistanceSelector(min_distance=1 * u.cm))
    else:
        template_events = week_data

    week_times = week_data.time
    week_orientation = sc_orientation_full.select_interval(week_times.min(), week_times.max())

    template.fill(template_events, week_orientation)

else:

    raise RuntimeError(f"bkg_template_source {bkg_template_source} is not supported.")


The template contains raw counts, which are noisy, so we smooth it:

- `psichi_fwhm`: a Gaussian on the sphere for every $\Psi\chi$ map;
- `phi_fwhm`: a Gaussian across neighboring $\phi$ bins of the $\Psi\chi$ maps;
- `time_counts`: a Gaussian along time for the rate, with a FWHM that depends on the statistics of each energy bin (about `time_counts` counts per FWHM), so high energies are integrated over longer times than low energies. It is applied to both the counts and the livetime, so the gaps (e.g. SAA, or a removed GRB window) are handled.

These values were chosen by cross-validation: we built a template from a random half of the events of weeks 3-5 (Distance >= 1 cm) and maximized the log-likelihood of the other half in 12 random 1-hour windows. Without any smoothing, some high-energy rate bins are empty and give a zero background probability to some events. The $\Psi\chi$ and $\phi$ smoothing gave the largest improvement; the statistics-dependent time smoothing was slightly better than a fixed 900 s FWHM, and left no event with zero probability. The 3-month template has more statistics than the one used for the cross-validation, so the optimal angular smoothing could be somewhat narrower.

In [ ]:
PSICHI_FWHM = 4 * u.deg
PHI_FWHM = 20 * u.deg
TIME_COUNTS = 500

template = template.smooth(psichi_fwhm=PSICHI_FWHM, phi_fwhm=PHI_FWHM, time_counts=TIME_COUNTS)

A look at the template: a $\Psi\chi$ map in the spacecraft frame (normalized to the total of the map) for one energy and $\phi$ bin, for each rocking bin.

In [ ]:
psichi_counts = template.psichi_counts
iem, iphi = 3, 6

em_edges = u.Quantity(psichi_counts.axes['Em'].edges, u.keV).value
phi_edges = u.Quantity(psichi_counts.axes['Phi'].edges, u.rad).to_value(u.deg)
nested = psichi_counts.axes['PsiChi'].is_nested

fig = plt.figure(figsize=(12, 4))

for k in range(2):
    pixels = psichi_counts.contents[k, iem, iphi]
    hp.mollview(pixels / pixels.sum(), nest=nested, sub=(1, 2, k + 1), fig=fig.number,
                title=f"Rocking bin {k}: Em in [{em_edges[iem]:.0f}, {em_edges[iem+1]:.0f}] keV, "
                      f"phi in [{phi_edges[iphi]:.0f}, {phi_edges[iphi+1]:.0f}] deg")


### Building the response

- `"nn"` builds `NFResponse` as in the other notebook (see there for a description of its
  `devices`/`compile_mode` arguments) and wraps it in `UnpolarizedNFFarFieldInstrumentResponseFunction`.
- `"hist_simple"` and `"hist_nn"` just load the pre-built histogram with
  `IRFRelativeHistUnpolarized.from_h5()` -- no compute pool or GPU involved.

In [ ]:
if irf_mode == "nn":

    rsp = NFResponse(
        path_to_model=rsp_path,
        area_batch_size=300_000,
        density_batch_size=100_000,
        devices=["cpu"],
        area_compile_mode=None,
        density_compile_mode=None,
        show_progress=True)

    irf = UnpolarizedNFFarFieldInstrumentResponseFunction(rsp)

elif irf_mode == "hist_simple":

    irf = IRFRelativeHistUnpolarized.from_h5(hist_path)

elif irf_mode == "hist_nn":

    irf = IRFRelativeHistUnpolarized.from_h5(hist_nn_path)


In [ ]:
psr = UnbinnedThreeMLPointSourceResponseIRFAdaptive(
    data=data,
    irf=irf,
    sc_history=sc_orientation,
    show_progress=True,
    force_energy_node_caching=True,
    reduce_memory=True)

psr.cache_batch_size = 100_000
psr.integration_batch_size = 100_000


The injected spectrum. The mock dataset uses the DC4 Crab, which is the sum of four components (the nebula, and the pulsar peaks P1, P2 and the bridge). Each is a tabulated spectral shape with a total flux in ph/cm$^2$/s over the energy range of the file (100 keV - 10 MeV), defined in the [cosi-sim source library](https://github.com/cositools/cosi-sim/tree/main/cosi_sim/Source_Library/DC4/sources/Galactic/Crab) (`Crab.source`). The injected spectrum is then
$\frac{\mathrm{d}N}{\mathrm{d}E} = \sum_i F_i \, s_i(E) / \int s_i(E') \mathrm{d}E'$.

In [ ]:
CRAB_SOURCE_URL = "https://raw.githubusercontent.com/cositools/cosi-sim/main/cosi_sim/Source_Library/DC4/sources/Galactic/Crab/"

# Spectral shape file and total flux [ph/cm2/s] of each component, from Crab.source in the same directory
CRAB_COMPONENTS = {"Crab_Ne_spec.dat": 0.033197515,
                   "Crab_P1_spec.dat": 0.002380673782066656,
                   "Crab_P2_spec.dat": 0.0032012384701172636,
                   "Crab_Brg_spec.dat": 0.0006280494007254569}

def read_spectrum_shape(filename):
    """Energy [keV] and (unnormalized) shape from the 'DP E value' lines of a MEGAlib spectrum file"""
    with urllib.request.urlopen(CRAB_SOURCE_URL + filename) as file:
        lines = file.read().decode().splitlines()

    energy, shape = np.array([[float(x) for x in line.split()[1:3]] for line in lines if line.startswith("DP")]).T

    return energy, shape

def trapezoid_integral(spectrum):
    energy, shape = spectrum
    return np.sum(np.diff(energy) * (shape[1:] + shape[:-1]) / 2)

crab_shapes = {name: read_spectrum_shape(name) for name in CRAB_COMPONENTS}

def injected_crab_spectrum(energy_keV):
    """Injected dN/dE [ph/cm2/s/keV]: each shape is normalized to its total flux over the file range"""
    return sum(flux * np.interp(energy_keV, *crab_shapes[name]) / trapezoid_integral(crab_shapes[name])
               for name, flux in CRAB_COMPONENTS.items())


We fit a power law, which is not exactly the injected spectrum.

In [ ]:
l = 184.56
b = -5.78

index = -2.2
piv = 100. * u.keV
K = 4.8e-4 / u.cm / u.cm / u.s / u.keV

spectrum = Powerlaw(index=index, K=K.value, piv=piv.value)

spectrum.index.min_value = -4
spectrum.index.max_value = -1

spectrum.K.unit = K.unit
spectrum.piv.unit = piv.unit

spectrum.index.delta = 0.01
spectrum.K.delta = 0.1 * K.value


In [ ]:
source = PointSource("Crab",
                     l=l,
                     b=b,
                     spectral_shape=spectrum)
model = Model(source)

In [ ]:
response = CachedUnbinnedThreeMLModelFolding(psr)

The data has no energy selection other than the energy range of the template, so we don't need to give an `energy_selection` to the background.

In [ ]:
bkg = FreeNormHistBackgroundDensity(
    data=data,
    sc_history=sc_orientation,
    template=template)

In [ ]:
expectation_density = SumExpectationDensity(response, bkg)

In [ ]:
like_fun = UnbinnedLikelihood(expectation_density)
cosi = ThreeMLPluginInterface('cosi', like_fun, response, bkg)

In [ ]:
bkg_norm = bkg.parameters['bkg_norm'].to_value(u.Hz)

cosi.bkg_parameter['bkg_norm'] = Parameter("bkg_norm",
                                           bkg_norm,
                                           unit = u.Hz,
                                           min_value=0,
                                           max_value=10 * bkg_norm,
                                           delta=0.01 * bkg_norm,
                                           )
print(f"The average background flux is {bkg_norm:.2f} Hz")

In [ ]:
plugins = DataList(cosi)
like = JointLikelihood(model, plugins, verbose=False) # You can enable debugging

### Initializing the Cache

The cache is initialized (can take some time)

In [ ]:
print(f"Data Events: {data.nevents}\nExpected Events: {expectation_density.expected_counts():.2f}\nRelative Deviation {100 * (expectation_density.expected_counts()/data.nevents - 1):.3f} %")


### Fitting

In [ ]:
like.fit()

The fitted background normalization, compared to the one predicted by the template:

In [ ]:
print(f"Fitted background rate: {cosi.bkg_parameter['bkg_norm'].value:.2f} Hz (template prediction: {bkg_norm:.2f} Hz)")

Now we can plot the result and compare it with the injected spectrum.

In [ ]:
results = like.results

parameters = {par.name: results.get_variates(par.path)
              for par in results.optimized_model["Crab"].parameters.values()
              if par.free}

results_err = results.propagate(results.optimized_model["Crab"].spectrum.main.shape.evaluate_at, **parameters)

In [ ]:
energy = np.geomspace(100*u.keV, 10*u.MeV).to_value(u.keV)

flux_lo = np.zeros_like(energy)
flux_median = np.zeros_like(energy)
flux_hi = np.zeros_like(energy)
flux_inj = injected_crab_spectrum(energy)

for i, e in enumerate(energy):
    flux = results_err(e)
    flux_median[i] = flux.median
    flux_lo[i], flux_hi[i] = flux.equal_tail_interval(cl=0.68)

In [ ]:
%matplotlib inline

In [ ]:
fig, ax = plt.subplots(figsize = (9, 6))

ax.plot(energy, energy**2 * flux_median, label = "Best fit")
ax.fill_between(energy, energy**2 * flux_lo, energy**2 * flux_hi, alpha = .5, label = "Best fit (errors)")
ax.plot(energy, energy**2 * flux_inj, color = 'black', ls = ":", label = "Injected")

ax.semilogx()
ax.semilogy()

ax.set_xlabel("Energy [keV]")
ax.set_ylabel(r"$E^2 \frac{\mathrm{d}N}{\mathrm{d}E}$ [keV cm$^{-2}$ s$^{-1}$]")

ax.legend();